# 03 - Faster R-CNN detector

Two-stage detector (torchvision, ResNet-50 FPN v2), trained on the same fused targets and the same image lists as YOLO. Different architecture -> different mistakes -> better fusion.

**Inputs:** output of `00_prep` + `vbd_utils`. **GPU required** (internet for the pretrained weights).

**Outputs:** `oof_frcnn_fold{k}.csv`, `test_frcnn_fold{k}.csv`, `frcnn_fold{k}.pt`.

**Two traps handled below:** torchvision resizes to 800 px by default (we force 1024), and label 0 is reserved for background (class_id + 1 in training, -1 at inference).

In [ ]:
# Import the shared utilities (upload vbd_utils.py as a Kaggle Dataset and attach it)
import os, sys, glob, warnings
def _locate(name, root='/kaggle/input'):
    for cur, dirs, files in os.walk(root):
        dirs[:] = [d for d in dirs if d not in ('png1024', 'images', 'labels', 'train', 'test')]
        if name in files:
            return cur
    raise FileNotFoundError(f'{name} not found - attach the vbd-utils dataset')
sys.path.insert(0, _locate('vbd_utils.py'))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import vbd_utils as U
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 30); pd.set_option('display.width', 200)
OUT = '/kaggle/working'

In [ ]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'   # avoids OOM from memory fragmentation on a T4
# ---------------- configuration ----------------
FOLDS = [0]
EPOCHS = 20
BATCH = 4
LR = 0.01
EVAL_EVERY = 2          # full validation (mAP) every N epochs - it takes a few minutes
NUM_WORKERS = 4
SEED = 42

In [ ]:
import cv2, gc, torch
import albumentations as A
from torch.utils.data import Dataset, DataLoader
from torchvision.models.detection import fasterrcnn_resnet50_fpn_v2, FasterRCNN_ResNet50_FPN_V2_Weights
from torchvision.models.detection.anchor_utils import AnchorGenerator
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from tqdm.auto import tqdm

torch.manual_seed(SEED); np.random.seed(SEED)
device = 'cuda'
PNG_DIR = U.find_dir('png1024')
meta = pd.read_csv(U.find_file('meta.csv'))
folds = pd.read_csv(U.find_file('folds.csv'), index_col=0).fold
targets = pd.read_csv(U.find_file('targets_fused.csv'))
test_ids = meta.loc[meta.split == 'test', 'image_id'].tolist()

## Model

In [ ]:
def build_model():
    m = fasterrcnn_resnet50_fpn_v2(weights=FasterRCNN_ResNet50_FPN_V2_Weights.DEFAULT,
                                   min_size=U.IMG_SIZE, max_size=U.IMG_SIZE,      # keep 1024 (default is 800)
                                   box_score_thresh=0.001, box_detections_per_img=300)
    # Smaller anchors for tiny nodules; 3 ratios kept so the pretrained RPN head still fits.
    # Ratio 0.5 (wide) covers Cardiomegaly boxes (height/width ~0.4) with IoU ~0.8.
    m.rpn.anchor_generator = AnchorGenerator(sizes=((16,), (32,), (64,), (128,), (256,)),
                                             aspect_ratios=((0.5, 1.0, 2.0),) * 5)
    m.roi_heads.box_predictor = FastRCNNPredictor(m.roi_heads.box_predictor.cls_score.in_features, 15)
    return m.to(device)

## Dataset

In [ ]:
aug = A.Compose([
    A.Affine(translate_percent=(-0.1, 0.1), scale=(0.8, 1.2), rotate=(-5, 5), p=0.7),   # no flips (anatomy)
    A.RandomBrightnessContrast(brightness_limit=0.1, contrast_limit=0.1, p=0.5)],
    bbox_params=A.BboxParams(format='pascal_voc', label_fields=['labels'], min_visibility=0.3))

by_img = targets.groupby('image_id')

class DetDS(Dataset):
    def __init__(self, ids, train=False):
        self.ids, self.train = list(ids), train
    def __len__(self):
        return len(self.ids)
    def __getitem__(self, i):
        iid = self.ids[i]
        img = cv2.imread(f'{PNG_DIR}/{iid}.png', 0)
        if not self.train:
            return torch.tensor(np.repeat(img[None], 3, 0) / 255.0, dtype=torch.float32), iid
        if iid in by_img.groups:
            t = by_img.get_group(iid)
            boxes = (t[U.BOX].values * U.IMG_SIZE).clip(0, U.IMG_SIZE).tolist()
            labels = (t.class_id.values + 1).tolist()
        else:
            boxes, labels = [], []                                  # normal image: background only
        r = aug(image=img, bboxes=boxes, labels=labels)
        img = r['image']
        keep = [(b, l) for b, l in zip(r['bboxes'], r['labels']) if b[2] - b[0] > 1 and b[3] - b[1] > 1]  # no degenerate boxes
        boxes, labels = [b for b, _ in keep], [l for _, l in keep]
        x = torch.tensor(np.repeat(img[None], 3, 0) / 255.0, dtype=torch.float32)   # model normalizes internally
        return x, {'boxes': torch.tensor(boxes, dtype=torch.float32).reshape(-1, 4),
                   'labels': torch.tensor(labels, dtype=torch.int64)}

collate = lambda b: tuple(zip(*b))

@torch.no_grad()
def predict(model, ids, bs=8):
    model.eval()
    dl = DataLoader(DetDS(ids), batch_size=bs, num_workers=NUM_WORKERS, collate_fn=collate)
    rows = []
    for imgs, iids in tqdm(dl, leave=False):
        with torch.autocast('cuda', dtype=torch.float16):
            outs = model([im.to(device) for im in imgs])
        for iid, o in zip(iids, outs):
            b = (o['boxes'].float() / U.IMG_SIZE).clamp(0, 1).cpu().numpy()
            for (x1, y1, x2, y2), l, s in zip(b, o['labels'].cpu().numpy() - 1, o['scores'].float().cpu().numpy()):
                rows.append((iid, int(l), float(s), x1, y1, x2, y2))
    del dl; gc.collect()          # shut down the worker processes before training forks new ones
    return pd.DataFrame(rows, columns=U.PRED_COLS)

## Training
Validation every `EVAL_EVERY` epochs with the competition metric (mAP over 14 findings, IoU 0.4); the best checkpoint is kept.

In [ ]:
def train_fold(k):
    tr_ids = open(U.find_file(f'det_train_fold{k}.txt')).read().split()
    va_ids = folds.index[folds == k]
    gt_va = U.build_gt(targets, va_ids)
    dl = DataLoader(DetDS(tr_ids, train=True), batch_size=BATCH, shuffle=True, num_workers=NUM_WORKERS,
                    collate_fn=collate, drop_last=True)
    model = build_model()
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.SGD(params, lr=LR, momentum=0.9, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, total_steps=EPOCHS * len(dl), pct_start=0.05)
    scaler = torch.amp.GradScaler('cuda')
    hist, best = [], -1
    for ep in range(EPOCHS):
        model.train(); losses = []
        for imgs, tgts in tqdm(dl, desc=f'fold {k} epoch {ep}', leave=False):
            imgs = [im.to(device) for im in imgs]
            tgts = [{key: v.to(device) for key, v in t.items()} for t in tgts]
            with torch.autocast('cuda', dtype=torch.float16):
                loss = sum(model(imgs, tgts).values())
            opt.zero_grad(); scaler.scale(loss).backward()
            scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(params, 10.0)
            scaler.step(opt); scaler.update(); sched.step()
            losses.append(loss.item())
        row = dict(epoch=ep, loss=np.mean(losses), map14=np.nan)
        if (ep + 1) % EVAL_EVERY == 0 or ep == EPOCHS - 1:
            row['map14'] = U.evaluate(gt_va, predict(model, va_ids), verbose=False)['map14']
            if row['map14'] > best:
                best = row['map14']; torch.save(model.state_dict(), f'{OUT}/frcnn_fold{k}.pt')
        hist.append(row)
        print(f"fold {k} ep {ep}: loss {row['loss']:.4f} | val mAP14 {row['map14']:.4f}")
    hist = pd.DataFrame(hist); hist.to_csv(f'{OUT}/frcnn_history_fold{k}.csv', index=False)
    return hist

for k in FOLDS:
    hist = train_fold(k)
    fig, ax = plt.subplots(1, 2, figsize=(14, 4))
    hist.plot(x='epoch', y='loss', ax=ax[0], title=f'fold {k}: training loss')
    hist.dropna().plot(x='epoch', y='map14', ax=ax[1], marker='o', title=f'fold {k}: validation mAP (14 findings)')
    plt.tight_layout(); plt.show()

    model = build_model(); model.load_state_dict(torch.load(f'{OUT}/frcnn_fold{k}.pt', map_location=device))
    predict(model, folds.index[folds == k]).to_csv(f'{OUT}/oof_frcnn_fold{k}.csv', index=False)
    predict(model, test_ids).to_csv(f'{OUT}/test_frcnn_fold{k}.csv', index=False)
    del model; torch.cuda.empty_cache()

## Evaluation report
Same report as YOLO, so the two detectors can be compared side by side (full comparison in notebook 04).

In [ ]:
oof = U.load_concat('oof_frcnn_fold*.csv')
eval_ids = folds.index[folds.isin(folds.loc[oof.image_id.unique()].unique())]
gt = U.build_gt(targets, eval_ids)
print('images evaluated:', len(eval_ids))
res = U.full_detection_report(gt, oof, PNG_DIR, 'Faster R-CNN (out-of-fold)', score_thr=0.2)

In [ ]:
for t in [0.05, 0.1, 0.2, 0.3, 0.5]:
    g = res['gt'][res['gt'].class_id != U.NO_FINDING]
    print(f'score >= {t:.2f}: recall {(g.det_score >= t).mean():.3f} | predictions per image {(oof.score >= t).sum() / len(eval_ids):.2f}')